# 🚢 Titanic - Machine Learning from Disaster
### Survival Prediction with Random Forest & Advanced Feature Engineering

This notebook documents the solution developed for the Kaggle competition **Titanic - Machine Learning from Disaster**.

**Achieved Result:** `Score: 0.80861` (~80.86% accuracy on the test set).

---

## 📌 Project Steps
1. **Data Loading & Imputation:** Handling missing values (`Age`, `Embarked`, `Fare`).
2. **Feature Engineering:**
   * Social Title extraction (`Title`) from passenger names.
   * Age categorization (`AgeGroup`) and fare quartiles (`FareGroup`).
   * Family size calculation (`FamilySize`) and solo traveler indicator (`IsAlone`).
3. **Family Survival Tracking:** Mapping family groups (Surname + Fare) to track shared family survival outcomes.
4. **Modeling:** **Random Forest Classifier** with controlled depth to prevent overfitting.

In [1]:
# --- 1. LIBRARIES & DATA LOADING ---
import pandas as pd
import warnings
from sklearn.ensemble import RandomForestClassifier

# Ignore Pandas deprecation warnings
warnings.filterwarnings('ignore')

# Load original train and test datasets
train_data = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Concatenate datasets temporarily to apply feature engineering consistently
full_data = pd.concat([train_data, test_data], axis=0, ignore_index=True)

print("Data loaded successfully!")
print(f"Total merged records: {full_data.shape[0]}")

Data loaded successfully!
Total merged records: 1309


## 🛠️ Feature Engineering & Data Cleaning
In this phase, missing values are handled and new categorical features are created:
* **Social Titles (`Title`):** Extracted from names to distinguish boys (*Master*), married women (*Mrs*), young women (*Miss*), adult men (*Mr*), and rare titles (*Rare*).
* **Grouping:** Ages converted into age groups (`AgeGroup`) and fares categorized into quartiles (`FareGroup`).

In [2]:
# --- 2. BASIC AND ADVANCED FEATURE ENGINEERING ---

# A) Handling Missing Values
full_data['Age'] = full_data['Age'].fillna(full_data['Age'].median())
full_data['Embarked'] = full_data['Embarked'].fillna('S')
full_data['Fare'] = full_data['Fare'].fillna(full_data['Fare'].median())

# B) Family Structure
full_data['FamilySize'] = full_data['SibSp'] + full_data['Parch'] + 1
full_data['IsAlone'] = (full_data['FamilySize'] == 1).astype(int)

# C) Extract Social Titles from Names
full_data['Title'] = full_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
full_data['Title'] = full_data['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 
                                             'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
full_data['Title'] = full_data['Title'].replace(['Mlle', 'Ms'], 'Miss')
full_data['Title'] = full_data['Title'].replace('Mme', 'Mrs')

title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
full_data['Title'] = full_data['Title'].map(title_mapping).fillna(0)

# D) Age Bins Categorization (AgeGroup)
full_data.loc[full_data['Age'] <= 12, 'AgeGroup'] = 0
full_data.loc[(full_data['Age'] > 12) & (full_data['Age'] <= 18), 'AgeGroup'] = 1
full_data.loc[(full_data['Age'] > 18) & (full_data['Age'] <= 50), 'AgeGroup'] = 2
full_data.loc[full_data['Age'] > 50, 'AgeGroup'] = 3

# E) Numeric Mapping & Fare Quartiles
full_data['Sex'] = full_data['Sex'].replace({'female': 1, 'male': 0})
full_data['Embarked'] = full_data['Embarked'].replace({'S': 0, 'C': 1, 'Q': 2})
full_data['FareGroup'] = pd.qcut(full_data['Fare'], 4, labels=[0, 1, 2, 3])

print("Data cleaning and basic feature engineering completed!")

Data cleaning and basic feature engineering completed!


## 👨‍👩‍👧‍👦 Family Survival Tracking
The primary technique responsible for boosting the model's performance. Family groups are identified by concatenating **Surname + Ticket Fare (`Fare`)**, allowing the model to track family members' historical survival outcomes.

In [3]:
# --- 3. FAMILY SURVIVAL TRACKING ---

full_data['Surname'] = full_data['Name'].apply(lambda x: x.split(',')[0].strip())
full_data['Family_ID'] = full_data['Surname'] + '_' + full_data['Fare'].astype(str)

# Initialize with neutral value (0.5) for solo travelers or unknown history
full_data['Family_Survival'] = 0.5 

# Analyze group survival within the same family
for family_id, group in full_data.groupby('Family_ID'):
    if len(group) > 1:
        for index, row in group.iterrows():
            other_members = group.drop(index)
            survived_members = other_members['Survived'].dropna()
            
            if len(survived_members) > 0:
                if survived_members.mean() == 1.0:
                    full_data.loc[index, 'Family_Survival'] = 1.0
                elif survived_members.mean() == 0.0:
                    full_data.loc[index, 'Family_Survival'] = 0.0

# Split back into train and test sets
train_set = full_data[full_data['Survived'].notnull()].copy()
test_set = full_data[full_data['Survived'].isnull()].copy()

print("Family survival mapping completed!")

Family survival mapping completed!


## 🤖 Model Training & Submission Generation
Training a **Random Forest Classifier** with 100 estimators and a maximum depth of 4 to ensure strong generalization. Finally, exporting predictions into `submission.csv`.

In [4]:
# --- 4. FEATURE SELECTION & TRAINING ---
features = ['Pclass', 'Sex', 'AgeGroup', 'FareGroup', 'Embarked', 'FamilySize', 'IsAlone', 'Title', 'Family_Survival']

X_train = train_set[features]
y_train = train_set['Survived'].astype(int)
X_test = test_set[features]

# Instantiate and train Random Forest
model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=1)
model.fit(X_train, y_train)

# --- 5. GENERATE SUBMISSION ---
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_set['PassengerId'].astype(int), 'Survived': predictions})
output.to_csv('submission.csv', index=False)

print("File 'submission.csv' successfully generated!")
output.head()

File 'submission.csv' successfully generated!


,PassengerId,Survived
891,892,0
892,893,1
893,894,0
894,895,0
895,896,1
